## 0. This checkout, not whatever is installed

`zou_lab_control_v2` is the one entry: importing it puts this checkout's eight layers
on the path, ahead of anything else.  It has to come **first** -- if a `zlc_*` module
was already imported from somewhere else, it refuses out loud rather than leaving two
copies in one kernel.  (If that happens: restart the kernel and run this cell first.)


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
while not (_here / 'zou_lab_control_v2').is_dir() and _here != _here.parent:
    _here = _here.parent
sys.path.insert(0, str(_here))

import zou_lab_control_v2

print('code in use:', zou_lab_control_v2.ROOT)


# zlc-runtime: from source data to consumable signals

This notebook is a repeatable tutorial. It builds an in-memory `zlc_data` source, then walks through coherent fronts, acquisition streams, node lifecycle, display cadence, and selection/fit derivation. Everything is local fake data: no hardware or server is required.

## 1. Prepare dependencies

This cell imports the small top-level runtime facade plus focused types used by the tutorial.

In [1]:
import numpy as np
from threading import Event
from zlc_data import (
    REPEAT, SCAN_POINT, AxisId, AxisSpec, BlockId, CellValidity,
    DataBlock, DatasetRevision, DatasetSchema, OwnedSnapshot,
    PointColumn, PointTable, StreamGenerationId, ValueSchema,
)
from zlc_runtime import (
    AcquisitionStream, FitEventValue, NodeHost, SelectionBridge,
    SelectionChange, SelectionRange, SelectionState, SignalDataPlane,
    SignalPublication, SignalValue, __version__,
)
from zlc_runtime.presentation import HarmonicClock, OwnerChannels
from zlc_runtime.dataset import MonitorCoverage
from zlc_runtime.dataset_output import (
    DatasetOutputDeclaration, FinalDatasetOutput, LiveDatasetOutput,
)
from zlc_runtime.streams import StreamId
print("runtime version:", __version__)
print("runtime and zlc_data are ready")

runtime version: 0.1.0
runtime and zlc_data are ready


## 2. Build an immutable source snapshot

`DataBlock` carries values, schema, and validity. `OwnedSnapshot` binds it to a revision and generation, so later source updates can replace the value without changing the data boundary.

In [2]:
repeat = AxisSpec(AxisId("repeat"), "repeat", REPEAT, 1, (0,))
x_column = PointColumn(AxisId("x"), "x", SCAN_POINT,
                       PointColumn.NUMERIC, (0.0, 1.0, 2.0))
source_schema = DatasetSchema(
    repeat, PointTable(3, (x_column,)), None,
    ValueSchema.scalar(np.dtype("float64"), "counts"),
)
def source_snapshot(revision):
    values = np.asarray([1.0, 2.0, 3.0]).reshape(1, 3, 1) + revision - 1
    block = DataBlock(BlockId(f"camera-{revision}"), DatasetRevision(revision),
                      values, CellValidity(np.ones((1, 3), dtype=bool)), source_schema)
    return OwnedSnapshot(block.ref(StreamGenerationId("camera-generation")), block)
def live_output(revision):
    declaration = DatasetOutputDeclaration("frame", "demo.camera.frame")
    return LiveDatasetOutput(declaration, source_snapshot(revision),
                             MonitorCoverage(3, 3, 0, False))
print("source schema:", source_schema.fingerprint[:12], "rows:", source_schema.point_table.row_count)

source schema: 416e9ad5ac76 rows: 3


## 3. Let SignalDataPlane own a live producer

A producer reserves its generation, attaches a live slot, and marks changes before the owner freezes a front. `set_front_signals` declares the coherent signals that readers care about.

In [3]:
class Camera:
    instance_id = "camera"
    dataset_output_declarations = (DatasetOutputDeclaration("frame", "demo.camera.frame"),)
    def signal_key(self, name):
        return f"camera/{name}"
class CameraSlot:
    notification_failure = None
    def __init__(self, state):
        self.state = state
    def freeze_live_outputs(self):
        return dict(self.state)
    def close(self):
        self.closed = True
camera = Camera()
camera_state = {"frame": live_output(1)}
camera_slot = CameraSlot(camera_state)
plane = SignalDataPlane()
plane.reserve(camera)
plane.attach(camera, camera_slot)
plane.set_front_signals({"camera/frame"})
plane.mark_changed(camera, camera_slot)
front = plane.freeze()
print("visible signals:", front.names())

visible signals: ('camera/frame',)


## 4. Read one coherent front

A front gives the reader both a `SignalValue` and its exact `SignalPublication`. The publication carries the event identity and keeps the value on the same-shot lineage.

In [4]:
frame = front.value("camera/frame")
publication = front.publication("camera/frame")
print("frame revision:", None if frame is None else frame.snapshot.ref.revision.value)
print("publication event:", None if publication is None else publication.event_ref)
print("frame values:", None if frame is None else frame.values.reshape(-1).tolist())
print("typed value:", isinstance(frame, SignalValue))
print("typed publication:", isinstance(publication, SignalPublication))

frame revision: 1
publication event: EventRef(stream_id=StreamId(value='camera'), generation=StreamGenerationId(value='1b33494b689e4272bf9cedfc7b0d0e8f'), sequence=1)
frame values: [1.0, 2.0, 3.0]
typed value: True
typed publication: True


## 5. Advance the source revision

Update the live slot, emit one owner wake, and freeze a new front. The producer generation remains stable while the publication sequence and data revision advance.

In [5]:
camera_state["frame"] = live_output(2)
plane.mark_changed(camera, camera_slot)
front = plane.freeze()
new_frame = front.value("camera/frame")
old_publication = publication
new_publication = front.publication("camera/frame")
print("new revision:", new_frame.snapshot.ref.revision.value)
print("same generation:", new_publication.event_ref.generation == old_publication.event_ref.generation)
print("new sequence:", new_publication.event_ref.sequence)

new revision: 2
same generation: True
new sequence: 2


## 6. Consume finite events with ExactReservation

An acquisition stream separates its producer from the one formal exact consumer. The cursor reads one delivery at a time, and the reservation acknowledgement advances the retention watermark.

In [6]:
class NumberContract:
    def snapshot(self, value):
        return int(value)
    def validate(self, value):
        if not isinstance(value, int):
            raise TypeError("numbers must be integers")
stream, producer = AcquisitionStream.create(StreamId("numbers"), NumberContract())
reservation = stream.reserve(total_events=2)
cursor = reservation.activate()
exact_consumer = object()
reservation.bind_consumer(exact_consumer, terminal=True)
monitor = stream.monitor()
follow = stream.follow()
for value in (10, 20):
    producer.emit(value, captured_at=float(value))
    delivery = cursor.next(timeout=1.0)
    print("exact delivery:", delivery.envelope.sequence, delivery.payload)
    reservation.acknowledge_delivery(delivery, exact_consumer)

exact delivery: 0 10
exact delivery: 1 20


## 7. Observe the same generation with monitor and follow

A monitor keeps an ordered observation queue. A follow tap receives every event published after subscription. Both use deadline-based reads, so the example does not guess with sleeps.

In [7]:
monitor_sequences = []
monitor_values = []
for _ in range(2):
    update = monitor.next(timeout=1.0)
    monitor_sequences.append(update.envelope.sequence)
    monitor_values.append(update.envelope.payload)
follow_values = [follow.next(timeout=1.0).payload for _ in range(2)]
eos = producer.finish()
reservation.complete_consumer(eos, exact_consumer)
print("monitor:", monitor_sequences, monitor_values)
print("follow:", follow_values)
print("end sequence:", eos.end_sequence)

monitor: [0, 1] [10, 20]
follow: [10, 20]
end sequence: 2


## 8. Run a finite node with NodeHost

A finite node publishes a FINAL Dataset through its execution context. The host owns the worker and lifecycle; an Event delivers wakeups before `poll` observes the result.

In [8]:
wake = Event()
result_declaration = DatasetOutputDeclaration("result", "demo.finite.result")
class FiniteNode:
    kind = "finite"
    instance_id = "finite-demo"
    dataset_output_declarations = (result_declaration,)
    def execute(self, context):
        context.publish_final({"result": FinalDatasetOutput(
            result_declaration, source_snapshot(2))})
        return {"message": "done"}
host = NodeHost(FiniteNode(), plane, wake.set)
host.start()
while not host.terminal:
    wake.wait(1.0)
    wake.clear()
    host.poll()
print("node phase:", host.observation.phase)
print("node result:", host.final_result)
host.shutdown()

node phase: done
node result: {'message': 'done'}


## 9. Combine wake channels with display cadence

`OwnerChannels` coalesces lifecycle and surface notifications into one owner turn. `HarmonicClock` aligns legal panel intervals to a shared display cadence without interpreting Dataset values.

In [9]:
class WakeSink:
    def __init__(self):
        self.calls = 0
    def request_owner_wake(self):
        self.calls += 1
sink = WakeSink()
channels = OwnerChannels(sink)
channels.notify_lifecycle()
channels.notify_surface()
turn = channels.take()
clock = HarmonicClock((10, 20, 40), default_ms=20)
clock.rebase((40, 20))
elapsed = clock.advance()
print("owner turn:", turn, "wake calls:", sink.calls)
print("clock:", elapsed, "group due:", clock.group_due(elapsed, (20, 40)))
channels.close()

owner turn: OwnerTurn(lifecycle=True, data=False, surface=True) wake calls: 2
clock: 20 group due: False


## 10. Prepare a pure numeric SelectionBridge event source

A plot or UI adapter only needs to convert its own events into the runtime's numeric table. This fake source implements the two callback protocols and can be replaced by a real adapter.

In [10]:
class FitEvents:
    def __init__(self):
        self.selection_callbacks, self.fit_callbacks = [], []
        self.current = None
    def subscribe_selection(self, callback):
        self.selection_callbacks.append(callback)
        return lambda: self.selection_callbacks.remove(callback)
    def subscribe_fit(self, callback):
        self.fit_callbacks.append(callback)
        return lambda: self.fit_callbacks.remove(callback)
    def selector_data(self, _kind):
        return self.current
    def emit_fit(self, event):
        for callback in tuple(self.fit_callbacks): callback(event)
    def emit_selection(self, change, state):
        self.current = state
        for callback in tuple(self.selection_callbacks): callback(change, state)
events = FitEvents()
plane.set_front_signals({"camera/frame", "@logic/tutorial/fit_center"})
bridge = SelectionBridge(plane, "camera/frame", events, events, bridge_id="tutorial")
bridge.start()
print("bridge started:", bridge.started)

bridge started: True


## 11. Publish a facet fit table

A fit event is a pure numeric table with an optional sample axis. For text facets, numeric point coordinates are indices while the original labels remain available as a TEXT point column.

In [11]:
events.emit_fit(FitEventValue(
    parameter_names=("center",),
    parameter_units={"center": "pixel"},
    parameter_values={"center": np.asarray([2.0, 2.5])},
    parameter_errors={"center": np.asarray([0.1, 0.2])},
    success=np.asarray([True, True]),
    sample_axis_name="facet", sample_coordinates=np.asarray([0.0, 1.0]),
    sample_unit="", sample_labels=("alpha", "beta"),
    source_revision=2, batch_revision=1,
))
fit_front = plane.freeze()
fit_value = fit_front.value("@logic/tutorial/fit_center")
columns = fit_value.snapshot.block.schema.point_table.columns
print("fit values:", fit_value.values.reshape(-1).tolist())
print("point columns:", [(column.name, column.values) for column in columns])
print("derived revision:", fit_value.snapshot.ref.revision.value)

fit values: [2.0, 2.5]
point columns: [('facet', (0, 1)), ('facet_label', ('alpha', 'beta'))]
derived revision: 1


## 12. Derive an ROI from the bound source snapshot

A selection event carries only ranges and facet conditions. The bridge still slices its bound source publication, preserving the source lineage for the derived ROI.

In [12]:
plane.set_front_signals({
    "camera/frame", "@logic/tutorial/fit_center", "@logic/tutorial/roi_value",
})
selection = SelectionState(
    "curve", "x_range", (SelectionRange("x", 0.0, 1.0),), revision=1
)
events.emit_selection(SelectionChange.COMMITTED, selection)
roi_front = plane.freeze()
roi = roi_front.value("@logic/tutorial/roi_value")
print("ROI mean:", roi.values.reshape(-1).tolist())
print("ROI event:", roi_front.publication("@logic/tutorial/roi_value").event_ref)

ROI mean: [2.5]
ROI event: EventRef(stream_id=StreamId(value='tutorial:selection:2'), generation=StreamGenerationId(value='0f99a645a6ce47e2b4cb215f8d7fea58'), sequence=1)


## 13. Keep source_revision and batch_revision separate

Repeated fits over the same source keep `source_revision` unchanged while every new publication advances `batch_revision`. The derived Dataset revision follows the publication counter.

In [13]:
events.emit_fit(FitEventValue(
    parameter_names=("center",), parameter_units={"center": "pixel"},
    parameter_values={"center": np.asarray([3.0, 3.5])},
    parameter_errors={"center": np.asarray([0.15, 0.25])},
    success=np.asarray([True, True]), sample_axis_name="facet",
    sample_coordinates=np.asarray([0.0, 1.0]), sample_unit="",
    sample_labels=("alpha", "beta"), source_revision=2, batch_revision=2,
))
second_fit = plane.freeze().value("@logic/tutorial/fit_center")
print("source stays:", 2, "batch advances to:", second_fit.snapshot.ref.revision.value)
print("new values:", second_fit.values.reshape(-1).tolist())

source stays: 2 batch advances to: 2
new values: [3.0, 3.5]


## 14. Close tutorial resources explicitly

Runtime owners have explicit lifetimes. Close the bridge, taps, source generation, and plane in teardown so the pattern can move directly into an application or test fixture.

In [14]:
bridge.close()
plane.retire(camera)
monitor.close()
follow.close()
plane.close()
print("closed: bridge, streams, camera generation, and signal plane")

closed: bridge, streams, camera generation, and signal plane
